# Stage 9 — Debit spread scale-up (218 stocks + 8 indices, 1H)

3-stock validation (BHARTIARTL, RELIANCE, NIFTY) showed 2/3 with a real edge
and 1 near-breakeven — encouraging but too thin a sample to trust. Round 1
of the full-universe run (214 instruments, 2024-11-15 to now, ~21 months)
gave 59.3% of instruments a positive avg R, but per-instrument trade counts
were thin (30-42 for the top names) since this signal only fires ~1-2x/month
per stock.

**This run extends the window to 2023-01-01 (~31 months)** for a larger
per-instrument sample — a moderate extension, not the full ~4yr window used
for the Supertrend strategies, since this is a short-term momentum signal
and reaching too far back risks mixing in an older, less-representative
volatility regime (see README assumptions log for the full reasoning).
force_refresh=True this run, since the window changed for every instrument.

New: `src/debit_spread_scale_pipeline.py` — parallel to `scale_pipeline.py` +
`compare_stocks.py`, not a modification of either. Same discipline: resume
support, isolated failure handling, rate-limit delay.

Uses the warm-up-buffered fetch window (a ~6-week buffer before the
intended analysis start date) so early
EMA/ADX instability doesn't contaminate signals.

This will take longer than round 1 (~31 months of 1H data vs ~21 — more API
chunks per instrument) — resume support means it's safe to re-run if
interrupted partway.

In [1]:
import sys
sys.path.append("..")

from datetime import datetime
from src.auth import get_kite
from src.instruments import get_fno_stock_list, build_nse_token_lookup, get_index_token
from src.debit_spread_scale_pipeline import run_all_instruments, build_comparison, save_comparison

kite = get_kite()
token_lookup = build_nse_token_lookup(kite)
stock_list = get_fno_stock_list(kite)
len(stock_list)

216

In [2]:
results = run_all_instruments(
    kite, token_lookup, stock_list,
    from_date=datetime(2023, 1, 1),
    to_date=datetime.now(),
    force_refresh=True
)
results

[1/216] 360ONE — processing...
Fetching 2023-01-01 to 2024-01-31...
Fetching 2024-02-01 to 2025-03-02...
Fetching 2025-03-03 to 2026-04-02...
Fetching 2026-04-03 to 2026-09-01...
Saved 6301 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/360ONE_60minute.csv
Saved 78 trades to /Users/hgmac/Documents/3_trading/Quant/exports/debit_spread/360ONE_trades.csv
[1/216] 360ONE — done (trades=77, win_rate=50.6, avg_r=0.19)
[2/216] ABB — processing...
Fetching 2023-01-01 to 2024-01-31...
Fetching 2024-02-01 to 2025-03-02...
Fetching 2025-03-03 to 2026-04-02...
Fetching 2026-04-03 to 2026-09-01...
Saved 6301 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/ABB_60minute.csv
Saved 59 trades to /Users/hgmac/Documents/3_trading/Quant/exports/debit_spread/ABB_trades.csv
[2/216] ABB — done (trades=59, win_rate=45.8, avg_r=0.08)
[3/216] ABCAPITAL — processing...
Fetching 2023-01-01 to 2024-01-31...
Fetching 2024-02-01 to 2025-03-02...
Fetching 2025-03-03 to 2026-04-02...
Fetching 2026

,total_trades,closed_trades,open_trades,win_rate_pct,avg_r_achieved,median_r_achieved,exit_reason_counts,median_candles_held,symbol,status,error
0,78.0,77.0,1.0,50.6,0.19,0.15,"{'STOP': 32, 'TARGET': 28, 'TIME': 17}",7.0,360ONE,SUCCESS,
1,59.0,59.0,0.0,45.8,0.08,-1.00,"{'STOP': 31, 'TARGET': 22, 'TIME': 6}",6.0,ABB,SUCCESS,
2,74.0,74.0,0.0,39.2,-0.13,-1.00,"{'STOP': 42, 'TARGET': 20, 'TIME': 12}",6.0,ABCAPITAL,SUCCESS,
3,71.0,71.0,0.0,52.1,0.26,0.17,"{'TARGET': 30, 'STOP': 30, 'TIME': 11}",5.0,ADANIENSOL,SUCCESS,
4,68.0,68.0,0.0,39.7,-0.08,-1.00,"{'STOP': 37, 'TARGET': 19, 'TIME': 12}",6.0,ADANIENT,SUCCESS,
...,...,...,...,...,...,...,...,...,...,...,...
211,59.0,59.0,0.0,49.2,0.12,-0.34,"{'STOP': 29, 'TARGET': 22, 'TIME': 8}",7.0,VOLTAS,SUCCESS,
212,32.0,32.0,0.0,53.1,0.19,0.04,"{'STOP': 14, 'TARGET': 13, 'TIME': 5}",5.5,WAAREEENER,SUCCESS,
213,72.0,72.0,0.0,50.0,0.18,0.04,"{'STOP': 34, 'TARGET': 30, 'TIME': 8}",6.0,WIPRO,SUCCESS,
214,84.0,84.0,0.0,32.1,-0.20,-1.00,"{'STOP': 55, 'TARGET': 25, 'TIME': 4}",5.0,YESBANK,SUCCESS,


## Indices — same 6 as the rest of this project (Stage 5), separate token lookup

In [3]:
from src.debit_spread_scale_pipeline import process_instrument

indices = [
    {"symbol": "NIFTY", "tradingsymbol": "NIFTY 50", "exchange": "NSE"},
    {"symbol": "BANKNIFTY", "tradingsymbol": "NIFTY BANK", "exchange": "NSE"},
    {"symbol": "SENSEX", "tradingsymbol": "SENSEX", "exchange": "BSE"},
    {"symbol": "NIFTYNXT50", "tradingsymbol": "NIFTY NEXT 50", "exchange": "NSE"},
    {"symbol": "MIDCPNIFTY", "tradingsymbol": "NIFTY MID SELECT", "exchange": "NSE"},
    {"symbol": "FINNIFTY", "tradingsymbol": "NIFTY FIN SERVICE", "exchange": "NSE"},
]

index_results = []
for idx in indices:
    token = get_index_token(kite, idx["tradingsymbol"], idx["exchange"])
    summary = process_instrument(idx["symbol"], token, kite,
                                  from_date=datetime(2023, 1, 1), to_date=datetime.now())
    summary["status"] = "SUCCESS"
    summary["error"] = ""
    index_results.append(summary)
    print(f"{idx['symbol']} — refreshed")

import pandas as pd
index_results_df = pd.DataFrame(index_results)
index_results_df

Fetching 2023-01-01 to 2024-01-31...
Fetching 2024-02-01 to 2025-03-02...
Fetching 2025-03-03 to 2026-04-02...
Fetching 2026-04-03 to 2026-09-01...
Saved 6322 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/NIFTY_60minute.csv
Saved 61 trades to /Users/hgmac/Documents/3_trading/Quant/exports/debit_spread/NIFTY_trades.csv
NIFTY — refreshed
Fetching 2023-01-01 to 2024-01-31...
Fetching 2024-02-01 to 2025-03-02...
Fetching 2025-03-03 to 2026-04-02...
Fetching 2026-04-03 to 2026-09-01...
Saved 6322 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/BANKNIFTY_60minute.csv
Saved 59 trades to /Users/hgmac/Documents/3_trading/Quant/exports/debit_spread/BANKNIFTY_trades.csv
BANKNIFTY — refreshed
Fetching 2023-01-01 to 2024-01-31...
Fetching 2024-02-01 to 2025-03-02...
Fetching 2025-03-03 to 2026-04-02...
Fetching 2026-04-03 to 2026-09-01...
Saved 6322 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/SENSEX_60minute.csv
Saved 50 trades to /Users/hgmac/Documents/3_trad

,total_trades,closed_trades,open_trades,win_rate_pct,avg_r_achieved,median_r_achieved,exit_reason_counts,median_candles_held,symbol,status,error
0,61,61,0,52.5,0.29,0.42,"{'TARGET': 28, 'STOP': 26, 'TIME': 7}",6.0,NIFTY,SUCCESS,
1,59,59,0,47.5,0.15,-0.36,"{'STOP': 27, 'TARGET': 23, 'TIME': 9}",6.0,BANKNIFTY,SUCCESS,
2,50,50,0,50.0,0.19,0.08,"{'STOP': 23, 'TARGET': 19, 'TIME': 8}",6.0,SENSEX,SUCCESS,
3,70,70,0,47.1,0.12,-1.00,"{'STOP': 36, 'TARGET': 27, 'TIME': 7}",5.5,NIFTYNXT50,SUCCESS,
4,86,86,0,47.7,0.15,-0.80,"{'STOP': 43, 'TARGET': 34, 'TIME': 9}",5.0,MIDCPNIFTY,SUCCESS,
5,74,74,0,44.6,0.05,-1.00,"{'STOP': 38, 'TARGET': 27, 'TIME': 9}",6.0,FINNIFTY,SUCCESS,


## Aggregate comparison — the actual verdict

In [4]:
all_results = pd.concat([results, index_results_df], ignore_index=True)
comparison = build_comparison(all_results)
save_comparison(comparison)
comparison

Saved debit spread comparison of 216 instruments to /Users/hgmac/Documents/3_trading/Quant/exports/DEBIT_SPREAD_COMPARISON.xlsx


,symbol,total_trades,closed_trades,win_rate_pct,avg_r_achieved,median_r_achieved,median_candles_held
0,SWIGGY,29.0,29.0,65.5,0.54,1.50,6.0
1,POWERINDIA,68.0,68.0,63.2,0.52,1.50,5.0
2,ATHERENERG,24.0,24.0,62.5,0.50,1.50,5.0
3,CDSL,68.0,68.0,57.4,0.41,1.46,6.0
4,HAL,69.0,69.0,56.5,0.38,1.05,7.0
...,...,...,...,...,...,...,...
211,YESBANK,84.0,84.0,32.1,-0.20,-1.00,5.0
212,MARICO,76.0,76.0,35.5,-0.23,-1.00,7.0
213,MPHASIS,85.0,84.0,34.5,-0.23,-1.00,6.0
214,BRITANNIA,73.0,73.0,31.5,-0.26,-1.00,6.0


In [5]:
# The actual answer to "does signal quality hold up": across every instrument
# with enough trades to mean anything (>= 15 closed trades, matching the
# minimum this project already uses in reversion_test.py), what fraction
# show a positive avg_r_achieved, and what does the distribution look like?
enough_trades = comparison[comparison["closed_trades"] >= 15]

print(f"Instruments processed: {len(comparison)}")
print(f"Instruments with >= 15 closed trades: {len(enough_trades)}")
print(f"Fraction with positive avg_r_achieved: "
      f"{(enough_trades['avg_r_achieved'] > 0).mean() * 100:.1f}%")
print()
print("avg_r_achieved distribution:")
print(enough_trades["avg_r_achieved"].describe())
print()
print("win_rate_pct distribution:")
print(enough_trades["win_rate_pct"].describe())

Instruments processed: 216
Instruments with >= 15 closed trades: 216
Fraction with positive avg_r_achieved: 69.0%

avg_r_achieved distribution:
count    216.000000
mean       0.075509
std        0.142567
min       -0.290000
25%       -0.030000
50%        0.080000
75%        0.170000
max        0.540000
Name: avg_r_achieved, dtype: float64

win_rate_pct distribution:
count    216.000000
mean      45.253241
std        5.823999
min       31.500000
25%       41.050000
50%       44.950000
75%       49.200000
max       65.500000
Name: win_rate_pct, dtype: float64
